# 2 · LangGraph — stateful agent
A looping ReAct agent with tools + memory (checkpointer).

In [ ]:
# Bootstrap: make the repo root importable so `import config` works from notebooks/
import sys, os
sys.path.insert(0, os.path.abspath(".."))
from config import assert_key
assert_key()
print("Gateway ready.")

In [ ]:
from config import get_langchain_llm
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

llm = get_langchain_llm()

@tool
def inventory(sku: str) -> int:
    'Units in stock for a product SKU (scout/hauler/sentinel).'
    return {"scout": 12, "hauler": 3, "sentinel": 0}.get(sku.lower(), 0)

@tool
def price(sku: str) -> int:
    'List price in USD for a product SKU.'
    return {"scout": 18000, "hauler": 42000, "sentinel": 30000}.get(sku.lower(), 0)

agent = create_react_agent(llm, tools=[inventory, price], checkpointer=MemorySaver())
cfg = {"configurable": {"thread_id": "nb-thread-1"}}

def ask(q):
    print("USER:", q)
    out = agent.invoke({"messages": [("user", q)]}, cfg)
    print("AGENT:", out["messages"][-1].content, "\n")

### Watch the loop + the checkpointer at work
The 2nd question has no subject — memory fills it in.

In [ ]:
ask("How many Hauler units do we have, and what's the total value of that stock?")
ask("And what about the Scout?")

## 🧪 Your turn
1. Add a `discount(sku, qty)` tool (e.g. 20% off above 50 units) and ask a multi-step question.
2. Change `thread_id` to a new value and re-ask *"And what about the Scout?"* — confirm memory does **not** carry across threads.

In [ ]:
# your code here